# 01 - Signal (Q2)

Text -> target with product/vendor tokens masked, against a majority-class baseline.

**Pass (pre-registered `gates.q2`):** masked macro-F1 >= majority + 0.15 (stratified 5-fold).
If the vendor-grouped split keeps < +0.05 over majority, the signal is vendor-specific.

In [1]:
import pandas as pd
from src.config import load_config, data_path

pd.set_option("display.width", 140)
cfg = load_config()
interim, processed = data_path(cfg, "interim"), data_path(cfg, "processed")

In [2]:
import json

results = {t: json.loads((processed / f"results_{t}.json").read_text()) for t in cfg["targets"]}
rows = []
for target, r in results.items():
    for name, run in r["signal"].items():
        rows.append({"target": target, "run": name, "macro_f1": run["macro_f1_mean"],
                     "sd": run["macro_f1_sd"], "accuracy": run["accuracy_mean"], "n": run["n"]})
pd.DataFrame(rows).round(3)

,target,run,macro_f1,sd,accuracy,n
0,sector,majority,0.129,0.000,0.634,1990
1,sector,vendor_lookup,0.773,0.014,0.820,1990
2,sector,masked,0.569,0.018,0.726,1990
3,sector,unmasked,0.735,0.029,0.796,1990
4,sector,majority_grouped,0.125,0.008,0.604,1990
5,sector,vendor_lookup_grouped,0.125,0.008,0.604,1990
6,sector,masked_grouped,0.222,0.016,0.391,1990
7,sector,unmasked_grouped,0.263,0.021,0.475,1990
8,time_to_kev,majority,0.222,0.001,0.500,522
9,time_to_kev,vendor_lookup,0.443,0.022,0.558,522


## Masking example

In [3]:
feats = pd.read_parquet(processed / "rows_sector.parquet")
for _, r in feats.sample(3, random_state=1).iterrows():
    print(r["cve_id"], r["label"]); print("  raw:   ", r["description"][:300]); print("  masked:", r["text_masked"][:300]); print()

CVE-2024-26885 Critical Manufacturing
  raw:    In the Linux kernel, the following vulnerability has been resolved:

bpf: Fix DEVMAP_HASH overflow check on 32-bit arches

The devmap code allocates a number hash buckets equal to the next power
of two of the max_entries value provided when creating the map. When
rounding up to the next power of two
  masked: In the xxproduct , the following vulnerability has been resolved: bpf: Fix DEVMAP_HASH overflow check on 32-bit arches The devmap code allocates a number hash buckets equal to the next power of two of the max_entries value provided when creating the map. When rounding up to the next power of two, th

CVE-2025-40810 Critical Manufacturing
  raw:    A vulnerability has been identified in Solid Edge SE2024 (All versions < V224.0 Update 14), Solid Edge SE2025 (All versions < V225.0 Update 6). The affected applications contains an out of bounds write vulnerability while parsing specially crafted PRT files. This could allow an attacker to 

## Per-class (masked text, stratified)

In [4]:
for target, r in results.items():
    print(target)
    display(pd.DataFrame(r["signal"]["masked"]["per_class"]).T.round(3))

sector


,precision,recall,f1-score,support
Commercial Facilities,0.432,0.422,0.427,83.0
Communications,0.534,0.667,0.593,93.0
Critical Manufacturing,0.853,0.803,0.827,1262.0
Energy,0.565,0.600,0.582,408.0
Healthcare and Public Health,0.527,0.718,0.608,110.0
Information Technology,0.524,0.324,0.400,34.0


time_to_kev


,precision,recall,f1-score,support
8-90d,0.409,0.481,0.442,131.0
<=7d,0.736,0.640,0.684,261.0
>90d,0.426,0.462,0.443,130.0


## Q2 decision

In [5]:
{t: r['q2'] for t, r in results.items()}

{'sector': {'gain_masked': 0.43966688638200624,
  'gain_unmasked': 0.6058644519955795,
  'gain_masked_grouped': 0.09713215276743967,
  'pass': True,
  'vendor_specific': False},
 'time_to_kev': {'gain_masked': 0.2986549537910991,
  'gain_unmasked': 0.33816295918418415,
  'gain_masked_grouped': 0.17013962003889893,
  'pass': True,
  'vendor_specific': False}}